<img src="https://courseware.decoded.com/images/decoded/logo-decoded.png" align="left" width="100px"/>

# Day 4 — Bitemporal loading and Airflow orchestration

<small>Data Creators › Day 4 — anchor notebook</small>

---
<small>&copy; 2026 Decoded Limited. All rights reserved. [decoded.com](https://decoded.com)</small>

<a id="how-to-use"></a>

## About this notebook

- Day 4's anchor: the agenda, then one numbered section per topic. Each section runs as a cycle: a demo, an activity or challenge, a shareback, a break.
- Every concept has its diagram; every demo has the cells that run it; every activity has its brief; every challenge points into the training repository.
- Run the **Setup** cell first. Cells that start with `!` run in a shell in this notebook's folder with the demo environment on `PATH`.
- 🖊️ Whiteboard lines are the questions the room works through together.
- The last section, **Questions from the room**, collects the day's questions and their answers as they come.

In [ ]:
# Setup: the demo environment on PATH, a sql() helper for showing results, a quick diagram helper
import os, sys, subprocess, textwrap
from pathlib import Path
from IPython.display import Image, Markdown, display

VENV = Path(os.environ.get("CG_DXP_VENV", Path.home() / ".venvs" / "cg-dxp"))
os.environ["PATH"] = ":".join([str(VENV / "bin"), str(Path.home() / ".local" / "bin"),
                               "/Applications/Docker.app/Contents/Resources/bin", "/opt/homebrew/bin", os.environ["PATH"]])
os.environ.setdefault("DBT_SEND_ANONYMOUS_USAGE_STATS", "false")
# The notebook's own folder, courseware/day4-*/, found from wherever the kernel is now, so this cell can be
# run again after a %cd into a demo folder. It also changes back to that folder.
HERE = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "day4_anchor.ipynb").exists()), None)
assert HERE is not None, "Open this notebook from its own folder (courseware/day4-*)"
os.chdir(HERE)
COMMON = HERE.parent / "common"
assert COMMON.exists(), "Open this notebook from its own folder (courseware/day4-*)"

def quick_diagram(mermaid: str, name: str = "scratch"):
    """Render a Mermaid diagram typed live and show it inline (needs node; first use downloads mermaid-cli)."""
    src = HERE / f"{name}.mmd"; out = HERE / f"{name}.png"
    src.write_text(textwrap.dedent(mermaid).strip() + "\n")
    cfg = COMMON / "diagrams" / "mermaid-config.json"
    r = subprocess.run(["npx", "-y", "-p", "@mermaid-js/mermaid-cli@11.17.0", "mmdc", "-i", str(src), "-o", str(out), "-c", str(cfg), "-b", "white", "-s", "2"],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-800:]); return
    display(Image(filename=str(out)))

def sql(query: str, db: Path, read_only: bool = True):
    """Run a query against a demo warehouse file and show it as a table."""
    import duckdb, pandas as pd
    pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
    con = duckdb.connect(str(db), read_only=read_only)
    try:
        return con.execute(query).df()
    finally:
        con.close()

print("✅ Day 4 setup:", subprocess.run(["dbt", "--version"], capture_output=True, text=True).stdout.split("\n")[1].strip() if subprocess.run(["which", "dbt"], capture_output=True).returncode == 0 else "dbt not on PATH: run common/env/setup-env.sh")

## Today

Morning: history (bitemporal loading) and orchestration (Airflow), both on yesterday's price sheet, then a long practical. Afternoon: the deployment picture, then two short exercises that turn a failed run into a decision and a note. Close by checking Friday's starting line.

| # | Section | What happens |
|---|---|---|
| 1 | **[Bitemporal loading](#s1)** | Demo: two clocks on the price sheet, three deliveries, three questions · Guest or volunteer: a real table · Shareback: which clock? |
| 2 | **[Airflow orchestration](#s2)** | Orientation: why orchestrate, and reading a DAG · Break · Demo: the challenge's steps 3 to 10 on the price sheet · Launch · Challenge 10: Airflow practical · Walkthrough: Phase 2 on the price sheet · Lunch · Shareback: one run, one log, one output check |
| 3 | **[CI/CD and release operations](#s3)** | Demo: GitOps, Harness, Datadog · Break · Activity 10: release decision · Activity 11: explain a failure and repair · Friday preparation |

<a id="s1"></a>

## 1. Bitemporal loading

### Demo — two clocks on the price sheet: three deliveries, three questions

<div style="border:2px solid #434343; border-radius:8px; margin:18px 0 12px; overflow:hidden;">
<div style="background:#434343; color:#fff; padding:8px 14px; font-weight:bold;">🧭 WHAT, WHY AND HOW</div>
<div style="background:#F3F3F3; padding:10px 14px; color:#1a1a1a; font-style:italic;">The vendor sends a wrong GBP rate on Wednesday and corrects it on Thursday. What was the rate, and what did Wednesday's report say?</div>
<div style="display:flex; gap:8px; padding:10px; background:#fff;"><div style="flex:1; min-width:0; background:#D0E0E3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHAT</div>Two clocks on every row: business time (the day the rate is for) and system time (when the warehouse believed it). Nothing is overwritten; a correction adds a row.</div><div style="flex:1; min-width:0; background:#FFF2CC; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHY</div>Audit, reproducible reports and 'as was' questions all need the second clock. Yesterday's price sheet overwrites: it cannot answer them.</div><div style="flex:1; min-width:0; background:#D9EAD3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">HOW</div>On yesterday's price sheet: a history model, a macro, three FX deliveries, then three questions.</div></div></div>

#### First: five questions, one picture each

<img src="../common/diagrams/png/d4-loading-strategies.png" alt="Four ways for new data to land in a table, and what each one remembers" style="max-height:340px; max-width:100%;">

**Think:** How can new data land in a table, and what does each way remember afterwards?

**Answers from the room:**

- 

<img src="../common/diagrams/png/d4-delivery-types.png" alt="What the source sends: everything, only the changes, or balances" style="max-height:380px; max-width:100%;">

**Think:** What does your source send each time: everything, or only what changed?

**Answers from the room:**

- 

<img src="../common/diagrams/png/d4-two-clocks.png" alt="Two clocks on every row: when it was true, and when we believed it" style="max-height:380px; max-width:100%;">

**Think:** What are the two clocks, and where does each one get its time from?

**Answers from the room:**

- 

<img src="../common/diagrams/png/d4-correction.png" alt="A correction, overwritten or kept" style="max-height:420px; max-width:100%;">

**Think:** The vendor corrects Wednesday's rate on Thursday. If we overwrite it, which question can nobody answer any more?

**Answers from the room:**

- 

<img src="../common/diagrams/png/d4-utc.png" alt="One run, two dates: the business date from the data, the system time from the load, in UTC" style="max-height:340px; max-width:100%;">

**Think:** A run at 03:00 Pacific on Thursday loads Wednesday's rates. Which date goes on which clock?

**Answers from the room:**

- 

#### From one clock to two, in tables: overwrite, SCD type 2, bitemporal

The same three deliveries in every table: the GBP rate for 3 November, a wrong rate for 4 November (1.628112), then the correction (1.268112).

**1. An ordinary table overwrites**

<div style="margin:10px 0 4px; font-weight:bold;">An ordinary table: one row per currency per day, overwritten when a correction arrives</div>
<table style="border-collapse:collapse; font-size:0.95em; margin:0;">
<tr><th style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#434343; color:#fff; text-align:left;">from_ccy</th><th style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#434343; color:#fff; text-align:left;">rate_dt</th><th style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#434343; color:#fff; text-align:left;">rate</th></tr>
<tr><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#FFFFFF;">GBP</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#FFFFFF;">3 Nov</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#FFFFFF;">1.267013</td></tr>
<tr><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#FFFFFF;">GBP</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#FFFFFF;">4 Nov</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F4CCCC; font-weight:bold;">1.268112</td></tr>
</table>
<div style="margin:4px 0 12px; font-style:italic;">On Wednesday evening the second row said 1.628112. The correction replaced it, and nothing in the table says so.</div>

**2. SCD type 2 keeps history, with one clock**

<div style="margin:10px 0 4px; font-weight:bold;">SCD type 2: a new row for each change, with one pair of dates</div>
<table style="border-collapse:collapse; font-size:0.95em; margin:0;">
<tr><th style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#434343; color:#fff; text-align:left;">from_ccy</th><th style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#434343; color:#fff; text-align:left;">rate</th><th style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#434343; color:#fff; text-align:left;">valid from</th><th style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#434343; color:#fff; text-align:left;">valid to</th><th style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#434343; color:#fff; text-align:left;">current</th></tr>
<tr><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F3F3F3;">GBP</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F3F3F3;">1.267013</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F3F3F3;">3 Nov</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F3F3F3;">3 Nov</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F3F3F3;">no</td></tr>
<tr><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F8DB59;">GBP</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F8DB59;">1.628112</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F8DB59;">4 Nov</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F8DB59;"></td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F8DB59;">yes</td></tr>
</table>
<div style="margin:4px 0 12px; font-style:italic;">After Tuesday's and Wednesday's deliveries. The second rate is the wrong one, though nobody knows that yet.</div>

**3. Where one clock runs out: the correction**

<div style="margin:10px 0 4px; font-weight:bold;">Choice 1: overwrite the wrong row</div>
<table style="border-collapse:collapse; font-size:0.95em; margin:0;">
<tr><th style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#434343; color:#fff; text-align:left;">from_ccy</th><th style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#434343; color:#fff; text-align:left;">rate</th><th style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#434343; color:#fff; text-align:left;">valid from</th><th style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#434343; color:#fff; text-align:left;">valid to</th><th style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#434343; color:#fff; text-align:left;">current</th></tr>
<tr><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F3F3F3;">GBP</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F3F3F3;">1.267013</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F3F3F3;">3 Nov</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F3F3F3;">3 Nov</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F3F3F3;">no</td></tr>
<tr><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#FFFFFF;">GBP</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F4CCCC; font-weight:bold;">1.268112</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#FFFFFF;">4 Nov</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#FFFFFF;"></td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#FFFFFF;">yes</td></tr>
</table>
<div style="margin:4px 0 12px; font-style:italic;">The table is right, and it has forgotten. Nobody can say what Wednesday's report used.</div>

<div style="margin:10px 0 4px; font-weight:bold;">Choice 2: add the correction as a new version</div>
<table style="border-collapse:collapse; font-size:0.95em; margin:0;">
<tr><th style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#434343; color:#fff; text-align:left;">from_ccy</th><th style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#434343; color:#fff; text-align:left;">rate</th><th style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#434343; color:#fff; text-align:left;">valid from</th><th style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#434343; color:#fff; text-align:left;">valid to</th><th style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#434343; color:#fff; text-align:left;">current</th></tr>
<tr><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F3F3F3;">GBP</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F3F3F3;">1.267013</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F3F3F3;">3 Nov</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F3F3F3;">3 Nov</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F3F3F3;">no</td></tr>
<tr><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#FFFFFF;">GBP</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#FFFFFF;">1.628112</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#FFFFFF;">4 Nov</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F4CCCC; font-weight:bold;">4 Nov</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F4CCCC; font-weight:bold;">no</td></tr>
<tr><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F4CCCC;">GBP</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F4CCCC;">1.268112</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F4CCCC;">5 Nov</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F4CCCC;"></td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F4CCCC;">yes</td></tr>
</table>
<div style="margin:4px 0 12px; font-style:italic;">The table now says the rate was 1.628112 on 4 November and changed on 5 November. It never changed: we were told the wrong number.</div>

**4. Bitemporal: two clocks on every row**

<div style="margin:10px 0 4px; font-weight:bold;">Bitemporal: the correction is its own row, for the same business date</div>
<table style="border-collapse:collapse; font-size:0.95em; margin:0;">
<tr><th style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#434343; color:#fff; text-align:left;">row</th><th style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#434343; color:#fff; text-align:left;">from_ccy</th><th style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#434343; color:#fff; text-align:left;">fx_rate</th><th style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#434343; color:#fff; text-align:left;">business from</th><th style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#434343; color:#fff; text-align:left;">business to</th><th style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#434343; color:#fff; text-align:left;">system from</th><th style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#434343; color:#fff; text-align:left;">system to</th><th style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#434343; color:#fff; text-align:left;">current</th></tr>
<tr><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F4CCCC;">C</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F4CCCC;">GBP</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F4CCCC;">1.628112</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F4CCCC;">4 Nov</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F4CCCC;"></td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F4CCCC;">Wed 18:00</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F4CCCC;">Thu 09:00</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F4CCCC;">no</td></tr>
<tr><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F8DB59;">D</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F8DB59;">GBP</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F8DB59;">1.268112</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F8DB59;">4 Nov</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F8DB59;"></td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F8DB59;">Thu 09:00</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F8DB59;"></td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F8DB59;">yes</td></tr>
</table>
<div style="margin:4px 0 12px; font-style:italic;">Both rows are for 4 November. Row C: what we believed from Wednesday evening to Thursday morning. Row D: what we believe now.</div>

**5. The whole table**

<div style="margin:10px 0 4px; font-weight:bold;">The same three deliveries in a bitemporal table: four rows, nothing overwritten</div>
<table style="border-collapse:collapse; font-size:0.95em; margin:0;">
<tr><th style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#434343; color:#fff; text-align:left;">row</th><th style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#434343; color:#fff; text-align:left;">from_ccy</th><th style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#434343; color:#fff; text-align:left;">fx_rate</th><th style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#434343; color:#fff; text-align:left;">business from</th><th style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#434343; color:#fff; text-align:left;">business to</th><th style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#434343; color:#fff; text-align:left;">system from</th><th style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#434343; color:#fff; text-align:left;">system to</th><th style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#434343; color:#fff; text-align:left;">current</th></tr>
<tr><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F3F3F3;">A</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F3F3F3;">GBP</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F3F3F3;">1.267013</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F3F3F3;">3 Nov</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F3F3F3;"></td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F3F3F3;">Tue 18:00</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F3F3F3;">Wed 18:00</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F3F3F3;">no</td></tr>
<tr><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#C9DAF8;">B</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#C9DAF8;">GBP</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#C9DAF8;">1.267013</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#C9DAF8;">3 Nov</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#C9DAF8;">3 Nov</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#C9DAF8;">Wed 18:00</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#C9DAF8;"></td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#C9DAF8;">no</td></tr>
<tr><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F4CCCC;">C</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F4CCCC;">GBP</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F4CCCC;">1.628112</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F4CCCC;">4 Nov</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F4CCCC;"></td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F4CCCC;">Wed 18:00</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F4CCCC;">Thu 09:00</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F4CCCC;">no</td></tr>
<tr><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F8DB59;">D</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F8DB59;">GBP</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F8DB59;">1.268112</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F8DB59;">4 Nov</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F8DB59;"></td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F8DB59;">Thu 09:00</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F8DB59;"></td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F8DB59;">yes</td></tr>
</table>
<div style="margin:4px 0 12px; font-style:italic;">An empty end means still open. Row C is the wrong rate: closed on Thursday morning, and kept.</div>

**6. The difference, side by side**

| | SCD type 2 | Bitemporal |
|---|---|---|
| Clocks | one | two |
| What was the value on date D? | yes | yes |
| What did we believe at time T? | no | yes |
| A correction | looks like a change, or is overwritten | kept as its own row |
| Cost | fewer rows | more rows |

**7. Which event moves which clock**

| What happened | Business clock | System clock |
|---|---|---|
| A new day's rate | a new period starts | a new row is believed from now |
| A correction | does not move | the old row is closed, a new row is believed from now |
| Late data: Monday's rate arrives on Thursday | says Monday | says Thursday |

#### One load, before and after: the correction arrives

<div style="margin:10px 0 4px; font-weight:bold;">Before the load: Wednesday evening. The wrong rate is current</div>
<table style="border-collapse:collapse; font-size:0.95em; margin:0;">
<tr><th style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#434343; color:#fff; text-align:left;">row</th><th style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#434343; color:#fff; text-align:left;">from_ccy</th><th style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#434343; color:#fff; text-align:left;">fx_rate</th><th style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#434343; color:#fff; text-align:left;">business from</th><th style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#434343; color:#fff; text-align:left;">business to</th><th style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#434343; color:#fff; text-align:left;">system from</th><th style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#434343; color:#fff; text-align:left;">system to</th><th style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#434343; color:#fff; text-align:left;">current</th></tr>
<tr><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F3F3F3;">A</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F3F3F3;">GBP</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F3F3F3;">1.267013</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F3F3F3;">3 Nov</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F3F3F3;"></td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F3F3F3;">Tue 18:00</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F3F3F3;">Wed 18:00</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F3F3F3;">no</td></tr>
<tr><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#C9DAF8;">B</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#C9DAF8;">GBP</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#C9DAF8;">1.267013</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#C9DAF8;">3 Nov</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#C9DAF8;">3 Nov</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#C9DAF8;">Wed 18:00</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#C9DAF8;"></td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#C9DAF8;">no</td></tr>
<tr><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F8DB59;">C</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F8DB59;">GBP</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F8DB59;">1.628112</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F8DB59;">4 Nov</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F8DB59;"></td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F8DB59;">Wed 18:00</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F8DB59;"></td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F8DB59;">yes</td></tr>
</table>

<div style="margin:10px 0 4px; font-weight:bold;">After the load: Thursday 09:00. One row updated, one row added</div>
<table style="border-collapse:collapse; font-size:0.95em; margin:0;">
<tr><th style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#434343; color:#fff; text-align:left;">row</th><th style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#434343; color:#fff; text-align:left;">from_ccy</th><th style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#434343; color:#fff; text-align:left;">fx_rate</th><th style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#434343; color:#fff; text-align:left;">business from</th><th style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#434343; color:#fff; text-align:left;">business to</th><th style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#434343; color:#fff; text-align:left;">system from</th><th style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#434343; color:#fff; text-align:left;">system to</th><th style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#434343; color:#fff; text-align:left;">current</th></tr>
<tr><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F3F3F3;">A</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F3F3F3;">GBP</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F3F3F3;">1.267013</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F3F3F3;">3 Nov</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F3F3F3;"></td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F3F3F3;">Tue 18:00</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F3F3F3;">Wed 18:00</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F3F3F3;">no</td></tr>
<tr><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#C9DAF8;">B</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#C9DAF8;">GBP</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#C9DAF8;">1.267013</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#C9DAF8;">3 Nov</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#C9DAF8;">3 Nov</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#C9DAF8;">Wed 18:00</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#C9DAF8;"></td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#C9DAF8;">no</td></tr>
<tr><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#FFFFFF;">C</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#FFFFFF;">GBP</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#FFFFFF;">1.628112</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#FFFFFF;">4 Nov</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#FFFFFF;"></td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#FFFFFF;">Wed 18:00</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F4CCCC; font-weight:bold;">Thu 09:00</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#F4CCCC; font-weight:bold;">no</td></tr>
<tr><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#D9EAD3;">D</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#D9EAD3;">GBP</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#D9EAD3;">1.268112</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#D9EAD3;">4 Nov</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#D9EAD3;"></td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#D9EAD3;">Thu 09:00</td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#D9EAD3;"></td><td style="border:1px solid #434343; padding:5px 10px; color:#1a1a1a; white-space:nowrap; background:#D9EAD3;">yes</td></tr>
</table>
<div style="margin:4px 0 12px; font-style:italic;">The load changed two cells of row C (its system clock is closed) and added row D. Rows A and B were not touched.</div>

#### The path for the next thirty minutes

<img src="../common/diagrams/png/d4-bt-path.png" alt="The path: eight steps from yesterday's price sheet to a table that remembers" style="max-height:380px; max-width:100%;">

*The path: eight steps from yesterday's price sheet to a table that remembers*

#### Step 1 — Start from yesterday's price sheet

In [ ]:
%cd {HERE}/demos/price_sheet
# Day 4's copy of yesterday's price sheet.

In [ ]:
!bash reset.sh
# The starting point: Day 3's finished project.

In [ ]:
!dbt seed
# The raw data first, as yesterday.

In [ ]:
!dbt build --exclude resource_type:seed
# The models and the tests, in graph order: yesterday's price sheet, built and passing.

#### Step 2 — The deliveries: three FX files from the vendor

In [ ]:
!dbt show --inline "select delivery_id, from_ccy, rate_dt as rate_for, cast(rate as varchar) as rate from demo.fx_deliveries order by delivery_id, from_ccy" --limit 10
# Three deliveries in one seed.

#### Step 3 — The history model: two clocks on every row

In [ ]:
%%writefile models/history/fx_rates_history.sql
-- fx_rates_history: every FX rate the vendor has sent, with two clocks. Each run loads one delivery.
--   incremental_strategy  our own strategy: the macro in macros/bitemporal_load.sql
--   bitemporal_key        what makes two rows "the same thing": the currency
--   full_refresh=false    never rebuild this table from scratch: that would erase the history
{{ config(materialized='incremental', incremental_strategy='bitemporal_load',
          bitemporal_key='from_ccy', full_refresh=false) }}

select
    from_ccy,
    cast(rate as decimal(18, 6))                  as fx_rate,
    md5(cast(rate as varchar))                    as dl_row_hash,            -- a fingerprint: a different hash means the rate changed
    cast(rate_dt as date)                         as dl_business_start_dt,   -- business clock: the day the rate is for
    cast(null as date)                            as dl_business_end_dt,
    cast(now() at time zone 'UTC' as timestamp)   as dl_system_start_ts,     -- system clock: when the warehouse learned it, in UTC
    cast(null as timestamp)                       as dl_system_end_ts,
    true                                          as dl_current_ind          -- true while both clocks are open
from {{ ref('fx_deliveries') }}
where delivery_id = '{{ var("delivery") }}'       -- one delivery per run: dbt run --vars "delivery: D1"

#### Step 4 — The macro: how a delivery lands in the table

<img src="../common/diagrams/png/d4-bt-macro.png" alt="What the macro does with each row of a new delivery" style="max-height:380px; max-width:100%;">

*What the macro does with each row of a new delivery*

In [ ]:
%%writefile macros/bitemporal_load.sql
{# bitemporal_load: how a new delivery lands in the history table.
   dbt has written the model's SELECT (the delivery) to a temporary table and calls this macro with:
     target = the history table, source = the delivery, cols = the column names.
   Steps 1 and 2 run inside the macro (run_query); step 3 is returned, and dbt runs it last. #}
{% macro get_incremental_bitemporal_load_sql(arg_dict) %}
  {%- set target = arg_dict["target_relation"] -%}
  {%- set source = arg_dict["temp_relation"] -%}
  {%- set key = config.require('bitemporal_key') -%}
  {%- set cols = arg_dict["dest_columns"] | map(attribute='name') | list -%}

  {#- 1. Close the SYSTEM clock of every current row whose rate the delivery changes. -#}
  {%- set close_sql -%}
    update {{ target }} as t
    set dl_system_end_ts = s.dl_system_start_ts, dl_current_ind = false
    from {{ source }} as s
    where t.{{ key }} = s.{{ key }} and t.dl_current_ind and t.dl_row_hash <> s.dl_row_hash
  {%- endset -%}
  {% do run_query(close_sql) %}

  {#- 2. A later business date: carry the old row forward, with its BUSINESS clock ended the day before. -#}
  {%- set carry_sql -%}
    insert into {{ target }} ({{ cols | join(', ') }})
    select
      {%- for c in cols %}
        {%- if c == 'dl_business_end_dt' %} cast(s.dl_business_start_dt - interval 1 day as date)
        {%- elif c == 'dl_system_start_ts' %} s.dl_system_start_ts
        {%- elif c == 'dl_system_end_ts' %} cast(null as timestamp)
        {%- elif c == 'dl_current_ind' %} false
        {%- else %} t.{{ c }}
        {%- endif %}{{ "," if not loop.last }}
      {%- endfor %}
    from {{ target }} as t
    join {{ source }} as s
      on s.{{ key }} = t.{{ key }}
     and t.dl_system_end_ts = s.dl_system_start_ts          -- the rows step 1 has just closed
     and s.dl_business_start_dt > t.dl_business_start_dt    -- only for a later day, not for a correction
  {%- endset -%}
  {% do run_query(carry_sql) %}

  {#- 3. Insert the delivery's rows, unless the same rate is already current (so a re-run changes nothing). -#}
  insert into {{ target }} ({{ cols | join(', ') }})
  select {{ cols | join(', ') }}
  from {{ source }} as s
  where not exists (
      select 1 from {{ target }} as t
      where t.{{ key }} = s.{{ key }} and t.dl_current_ind and t.dl_row_hash = s.dl_row_hash
  )
{% endmacro %}

#### Step 5 — Load 1: the first day

In [ ]:
!dbt run --select fx_rates_history --vars "delivery: D1"
# --vars passes a value into the model: which delivery to load.

In [ ]:
!dbt show --inline "select cast(fx_rate as varchar) as fx_rate, dl_business_start_dt as rate_for, dl_business_end_dt as rate_until, strftime(dl_system_start_ts, '%H:%M:%S') as believed_from, strftime(dl_system_end_ts, '%H:%M:%S') as believed_until, dl_current_ind as current from demo.fx_rates_history where from_ccy = 'GBP' order by dl_system_start_ts, dl_business_start_dt" --limit 10
# One row for GBP: the rate for 3 November, believed since load 1, open on both clocks.

<img src="../common/diagrams/png/d4-bt-load-1.png" alt="After load 1: one row, open on both clocks" style="max-height:200px; max-width:100%;">

*After load 1: one row, open on both clocks*

#### Step 6 — Load 2: a new day, with one wrong rate

In [ ]:
!dbt run --select fx_rates_history --vars "delivery: D2"
# The table exists now, so the macro runs: close, carry forward, insert.

In [ ]:
!dbt show --inline "select cast(fx_rate as varchar) as fx_rate, dl_business_start_dt as rate_for, dl_business_end_dt as rate_until, strftime(dl_system_start_ts, '%H:%M:%S') as believed_from, strftime(dl_system_end_ts, '%H:%M:%S') as believed_until, dl_current_ind as current from demo.fx_rates_history where from_ccy = 'GBP' order by dl_system_start_ts, dl_business_start_dt" --limit 10
# Three rows.

<img src="../common/diagrams/png/d4-bt-load-2.png" alt="After load 2: the old rate is kept for its own day, and the new rate is current" style="max-height:420px; max-width:100%;">

*After load 2: the old rate is kept for its own day, and the new rate is current*

#### Step 7 — Load 3: the vendor's correction

In [ ]:
!dbt run --select fx_rates_history --vars "delivery: D2-CORR"
# The same business date, a different rate: a correction.

In [ ]:
!dbt show --inline "select cast(fx_rate as varchar) as fx_rate, dl_business_start_dt as rate_for, dl_business_end_dt as rate_until, strftime(dl_system_start_ts, '%H:%M:%S') as believed_from, strftime(dl_system_end_ts, '%H:%M:%S') as believed_until, dl_current_ind as current from demo.fx_rates_history where from_ccy = 'GBP' order by dl_system_start_ts, dl_business_start_dt" --limit 10
# Four rows.

<img src="../common/diagrams/png/d4-bt-load-3.png" alt="After load 3: the wrong rate is closed and kept; the corrected rate is current" style="max-height:460px; max-width:100%;">

*After load 3: the wrong rate is closed and kept; the corrected rate is current*

#### Step 8 — Three questions, one table

<img src="../common/diagrams/png/d4-bt-questions.png" alt="Three questions, each answered by filtering on the clocks" style="max-height:340px; max-width:100%;">

*Three questions, each answered by filtering on the clocks*

In [ ]:
!dbt show --inline "select from_ccy, cast(fx_rate as varchar) as fx_rate, dl_business_start_dt as rate_for from demo.fx_rates_history where dl_current_ind order by from_ccy" --limit 10
# AS IS: what is true now.

In [ ]:
!dbt show --inline "select from_ccy, cast(fx_rate as varchar) as fx_rate, dl_business_start_dt as rate_for from demo.fx_rates_history where dl_business_start_dt <= date '2026-11-03' and coalesce(dl_business_end_dt, date '9999-12-31') >= date '2026-11-03' and dl_system_end_ts is null order by from_ccy" --limit 10
# AS OF 3 November, with today's knowledge: the business clock covers the date, the system clock is open.

In [ ]:
!dbt show --inline "select from_ccy, cast(fx_rate as varchar) as fx_rate, dl_business_start_dt as rate_for from demo.fx_rates_history where dl_system_start_ts < (select max(dl_system_start_ts) from demo.fx_rates_history) and coalesce(dl_system_end_ts, timestamp '9999-12-31') >= (select max(dl_system_start_ts) from demo.fx_rates_history) and dl_business_end_dt is null order by from_ccy" --limit 10
# AS WAS, just before the correction arrived: the system clock covers that moment.

In [ ]:
!dbt show --inline "select p.instrument_id, p.price_date, cast(p.price_native as varchar) as price_gbp, cast(h.fx_rate as varchar) as fx_rate, cast(round(p.price_native * h.fx_rate, 4) as varchar) as price_usd, case when h.dl_system_end_ts is null then 'as is' else 'as was' end as version from demo.stg_prices p join demo.fx_rates_history h on h.from_ccy = p.native_ccy and h.dl_business_start_dt = p.price_date where p.native_ccy = 'GBP' and p.price_date = date '2026-11-04' order by h.dl_system_start_ts" --limit 10
# The same price in dollars, twice: what Wednesday's price sheet said, and what it says now.

| The question | Which clock | The filter |
|---|---|---|
| What is current? | both open | `dl_current_ind` |
| What was true on date D, as we know it today? | business covers D, system open | `dl_business_start_dt <= D and coalesce(dl_business_end_dt, '9999-12-31') >= D and dl_system_end_ts is null` |
| What did the report say at time T? | system covers T | `dl_system_start_ts <= T and coalesce(dl_system_end_ts, '9999-12-31') > T` |
| Which versions did a correction replace? | same business start, later system start | order by `dl_system_start_ts` within the key |

In [ ]:
%cd {HERE}

### Guest or volunteer — a real bitemporal table

<div style="border:2px solid #F8DB59; border-radius:8px; margin:10px 0;">
<div style="background:#FFF2CC; color:#000; padding:8px 14px; font-weight:bold;">💭 GUEST OR VOLUNTEER BRIEF — ten minutes</div>
<div style="background:#fffef5; padding:12px 14px; color:#1a1a1a;">Show one real bitemporal table, then take two questions.</div></div>

- The DL columns, and one row's history ordered by system start.
- One change that created a version; one correction if you have one.
- Which strategy the model uses: full, incremental or balance.
- If a defect once created a version, show it: the table is an audit of what the code did.

### Shareback — which clock?

<div style="border:2px solid #F8DB59; border-radius:8px; margin:10px 0;">
<div style="background:#FFF2CC; color:#000; padding:8px 14px; font-weight:bold;">💭 SHAREBACK — which clock?</div>
<div style="background:#fffef5; padding:12px 14px; color:#1a1a1a;">Each table takes one question: say the clock, then the filter.</div></div>

- What was the GBP rate for 3 November?
- What rate did Wednesday morning's price sheet use for 4 November?
- When did we learn about the correction?
- Which rows does an auditor need to see?

<a id="s2"></a>

## 2. Airflow orchestration

### Orientation — why orchestrate, and reading a DAG

<div style="border:2px solid #434343; border-radius:8px; margin:18px 0 12px; overflow:hidden;">
<div style="background:#434343; color:#fff; padding:8px 14px; font-weight:bold;">🧭 WHAT, WHY AND HOW</div>
<div style="background:#F3F3F3; padding:10px 14px; color:#1a1a1a; font-style:italic;">Yesterday's price sheet builds when you type dbt build. Who types it at six in the morning?</div>
<div style="display:flex; gap:8px; padding:10px; background:#fff;"><div style="flex:1; min-width:0; background:#D0E0E3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHAT</div>Orchestration: deciding when the pipeline's parts run, in what order, with retries, logs and alerts.</div><div style="flex:1; min-width:0; background:#FFF2CC; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHY</div>A product's promises about delivery time and freshness are kept by the scheduler, not by the SQL.</div><div style="flex:1; min-width:0; background:#D9EAD3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">HOW</div>Yesterday's price sheet as an Airflow DAG, in the same ten steps as your challenge.</div></div></div>

#### 1 — Why orchestration

<img src="../common/diagrams/png/d4-why-orchestration.png" alt="Five things a hand-typed dbt build cannot do, and what Airflow gives you for each" style="max-height:400px; max-width:100%;">

*Five things a hand-typed dbt build cannot do, and what Airflow gives you for each*

#### The path: the ten steps of the challenge, on the price sheet

<img src="../common/diagrams/png/d4-airflow-path.png" alt="The path: read the DAG, then wire it, run it and break it" style="max-height:380px; max-width:100%;">

*The path: read the DAG, then wire it, run it and break it*

<img src="../common/diagrams/png/d4-ps-dag.png" alt="The DAG we will run: a gate, yesterday's models as tasks, a publish step" style="max-height:300px; max-width:100%;">

*The DAG we will run: a gate, yesterday's models as tasks, a publish step*

In [ ]:
%cd {HERE}/demos/airflow_local
# The local Airflow stack.

In [ ]:
!bash run.sh unwire
# The starting point: the DAG as learners get it.

In [ ]:
!sed -n '1,16p' dags/price_sheet/dags/price_sheet_daily.py
# Read the top of the file.

#### 2 — Anatomy of a DAG

<img src="../common/diagrams/png/d4-dag-anatomy.png" alt="A DAG is a Python file: the DAG(...) block, the tasks inside it, and >> between them" style="max-height:320px; max-width:100%;">

*A DAG is a Python file: the DAG(...) block, the tasks inside it, and >> between them*

In [ ]:
!sed -n '/^default_args/,/^) as dag:/p' dags/price_sheet/dags/price_sheet_daily.py
# default_args: what every task inherits.

In [ ]:
!python -c "import ast; ast.parse(open('dags/price_sheet/dags/price_sheet_daily.py').read()); print('parses OK')"
# The cheapest check: is the file valid Python?

In [ ]:
!astro dev run dags list-import-errors | grep -v "secrets backend"
# The real check: did Airflow load every DAG file?

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#EFEFEF; color:#1a1a1a; padding:8px 14px; font-weight:bold;">☕ BREAK &nbsp;·&nbsp; 15 min &nbsp;·&nbsp; back at 10:15</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">Questions go in the table at the end; the next part starts on time.</div></div>

### Demo — connect, gate, group, wire, schedule, retry, run, repair

<div style="border:2px solid #434343; border-radius:8px; margin:18px 0 12px; overflow:hidden;">
<div style="background:#434343; color:#fff; padding:8px 14px; font-weight:bold;">🧭 WHAT, WHY AND HOW</div>
<div style="background:#F3F3F3; padding:10px 14px; color:#1a1a1a; font-style:italic;">What is actually running when you type astro dev start?</div>
<div style="display:flex; gap:8px; padding:10px; background:#fff;"><div style="flex:1; min-width:0; background:#D0E0E3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHAT</div>A DAG processor, a scheduler, workers, a triggerer, an API server with the UI, and a metadata database: on your laptop as containers, or hosted by Astronomer.</div><div style="flex:1; min-width:0; background:#FFF2CC; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHY</div>Knowing the pieces explains the symptoms: an import error is the processor; a queued task is the scheduler; a failed task is a worker log.</div><div style="flex:1; min-width:0; background:#D9EAD3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">HOW</div>Steps 3 to 10 on the price sheet: one command per cell, and the UI at http://localhost:8080 beside it.</div></div></div>

![Airflow's components, locally and hosted](../common/diagrams/png/d4-airflow-components.png)

*Airflow's components, locally and hosted*

#### 3 — Connect: can dbt reach the warehouse from inside Airflow?

In [ ]:
!bash run.sh dbt price_sheet debug --connection | tail -3
# dbt debug, run inside the scheduler container, in the price sheet project.

In [ ]:
!bash run.sh seed 2>&1 | grep -E 'Done|rror'
# The vendor delivers: the raw feeds land in the warehouse.

#### 4 — The first task: the freshness gate

<img src="../common/diagrams/png/d4-freshness-gate.png" alt="The freshness gate: how old is the feed, and does the run carry on?" style="max-height:340px; max-width:100%;">

*The freshness gate: how old is the feed, and does the run carry on?*

In [ ]:
!grep -n -B1 -A4 'loaded_at_field' dags/price_sheet/dbt/models/sources.yml
# The contract for the feed: which column says when it arrived, when to warn, when to stop.

In [ ]:
!sed -n '/^def _check_source_freshness/,/^        )/p' dags/price_sheet/dags/price_sheet_daily.py
# The task's function: run dbt source freshness, and raise an error if it fails.

In [ ]:
!astro dev run tasks test price_sheet_daily source_freshness_check 2026-11-12 2>&1 | grep -E -o '(START|PASS|WARN|ERROR) freshness of [a-z_.]+|new_state=[a-z_]+'
# Run this one task on its own: no schedule, no waiting for the tasks before it.

#### 5 — The Cosmos task group: yesterday's models become tasks

<img src="../common/diagrams/png/d4-cosmos.png" alt="Cosmos reads the dbt project and makes one Airflow task per model, in dbt's order" style="max-height:340px; max-width:100%;">

*Cosmos reads the dbt project and makes one Airflow task per model, in dbt's order*

In [ ]:
!sed -n '/^    dbt_pipeline = DbtTaskGroup/,/^    )/p' dags/price_sheet/dags/price_sheet_daily.py
# The whole dbt project in one call.

In [ ]:
!astro dev run tasks list price_sheet_daily | grep -v "secrets backend"
# Five tasks named dbt_pipeline.<model>_run and one dbt_test: yesterday's models.

#### 6 — Wire the chain: the one line learners write

In [ ]:
!grep -n "TODO" dags/price_sheet/dags/price_sheet_daily.py || echo "no TODO left: the chain is already wired"
# Five tasks exist, and nothing connects them yet.

In [ ]:
!bash run.sh wire
# The TODO replaced by one line.

In [ ]:
!astro dev run dags reserialize > /dev/null 2>&1; echo 'Airflow has read the DAG files again'
# Ask Airflow to read the DAG files now, not at its next scan.

In [ ]:
!python -m pytest dags/price_sheet/tests -q -k chain
# The check that the chain is wired.

#### 7 — Scheduling: a cron in a time zone

<img src="../common/diagrams/png/d4-cron-syntax.png" alt="Reading a cron schedule: five fields, four symbols" style="max-height:460px; max-width:100%;">

*Reading a cron schedule: five fields, four symbols*

<img src="../common/diagrams/png/d4-cron.png" alt="The schedule: five cron fields, and a start date that carries the time zone" style="max-height:380px; max-width:100%;">

*The schedule: five cron fields, and a start date that carries the time zone*

In [ ]:
!astro dev run dags details price_sheet_daily 2>&1 | grep -i -E "timetable|max_active"
# The schedule as Airflow understood it.

#### 8 — Retries and backoff

<img src="../common/diagrams/png/d4-retries.png" alt="Retries: a task that fails waits and tries again, and only alerts a person when the retries run out" style="max-height:340px; max-width:100%;">

*Retries: a task that fails waits and tries again, and only alerts a person when the retries run out*

In [ ]:
!sed -n '/^default_args/,/^}/p' dags/price_sheet/dags/price_sheet_daily.py
# Set once, inherited by every task.

#### 9 — The first full run

In [ ]:
!astro dev run dags unpause price_sheet_daily | grep -v "secrets backend"
# A new DAG starts paused: Airflow knows it and will not run it.

In [ ]:
!astro dev run dags trigger price_sheet_daily 2>&1 | grep -c queued
# Ask for one run now, on top of the schedule.

In [ ]:
!bash run.sh wait price_sheet_daily
# Watch it in the UI while this polls: Grid (runs over time), Graph (the tasks of one run), then a task's Logs.

In [ ]:
!bash run.sh status price_sheet_daily
# The run, then the state of every task in it.

In [ ]:
!bash run.sh dbt price_sheet show --select fct_prices_usd --limit 5
# The proof is in the warehouse, not in the green boxes: the same rows a hand-typed dbt build produces.

#### 10 — Reading logs and re-running: break it, read it, fix it, clear it

<img src="../common/diagrams/png/d4-clear.png" alt="A failed run: read the log, fix the cause, then clear the failed task to re-run it and everything after it" style="max-height:340px; max-width:100%;">

*A failed run: read the log, fix the cause, then clear the failed task to re-run it and everything after it*

In [ ]:
!bash run.sh stale price_sheet_daily
# Break it on purpose: the vendor feed now looks two days old.

In [ ]:
!astro dev run dags trigger price_sheet_daily 2>&1 | grep -c queued

In [ ]:
!bash run.sh wait price_sheet_daily
# About a minute.

In [ ]:
!bash run.sh status price_sheet_daily | grep -E 'task_id|freshness|dbt_pipeline|publish'
# The gate failed; everything after it is upstream_failed.

In [ ]:
!bash run.sh fresh price_sheet_daily
# Fix the cause: the feed arrives.

In [ ]:
!bash run.sh wait price_sheet_daily
# After the Clear in the UI: the same run, green this time.

#### At CG: most DAGs come from a blueprint

![Astro blueprints: YAML in, a DAG out](../common/diagrams/png/d4-blueprints.png)

*Astro blueprints: YAML in, a DAG out*

In [ ]:
!sed -n '1,30p' ../../../../reference/DataCreators-client/dev-container/edo-training-data-products/data_products/.blueprints/dag_exec_check.py
# A platform blueprint from the vended repository: the Python learners never write, filled from YAML.

### Launch — from this demo to your challenge

<img src="../common/diagrams/png/d4-demo-to-challenge.png" alt="From this demo to the challenge: the same ten steps, on the Portfolio Analytics project" style="max-height:520px; max-width:100%;">

*From this demo to the challenge: the same ten steps, on the Portfolio Analytics project*

### Challenge 10 — Airflow practical

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#434343; color:#fff; padding:8px 14px; font-weight:bold;">🏆 CHALLENGE TIME &nbsp;—&nbsp; Airflow practical (challenge 03) &nbsp;·&nbsp; until 13:30, with lunch</div>
<div style="background:#f8f8f8; padding:12px 14px; color:#1a1a1a;">Wire, schedule and run the Portfolio Analytics DAG with Cosmos; Phase 2 is production operations. &nbsp;·&nbsp; <code>challenges/v1/03-airflow-orchestration/README.md</code></div></div>

**Prerequisites, in this order**

1. `mise run setup:databricks` done, in a terminal where `echo $DBT_CATALOG` prints a value.
2. `(cd challenges/v1/02-dbt-pipeline/dbt && dbt seed)`: the DAG reads the same four feeds, refreshed.
3. From the repo root, `astro dev start`, only after setup finished; the UI address is the one astro prints, not a fixed localhost:8080.

**Phase 1, about 1.5 hours**

1. 1 Why orchestration.
2. 2 Anatomy of a DAG.
3. 3 Connect to Databricks: `dbt debug`.
4. 4 The freshness gate: `astro dev run tasks test portfolio_analytics_daily source_freshness_check 2026-05-04`.
5. 5 The Cosmos `DbtTaskGroup`.
6. 6 Wire the chain, the one TODO: `start >> source_freshness_check >> dbt_pipeline >> publish_to_catalog >> end`.
7. 7 Scheduling: a timezone-aware cron.
8. 8 Retries and exponential backoff.
9. 9 First full run.
10. 10 Reading logs and re-running.

**Phase 2**

1. 1 The vendor-neutral alert transport.
2. 2 Tiered failure alerting.
3. 3 The P2 and P3 tiers: SLA miss and retry.
4. 4 Wire the callbacks and the SLA into the DAG.
5. 5 The freshness gate as a contract.
6. 6 The monitoring config.
7. 7 The runbook and the consumer guide.
8. 8 Structural tests for the DAG.
9. 9 Stretch: the `DatabricksSubmitRunOperator` variant.
10. 10 Capstone: a production-ready DAG.

**Gate.** `mise run challenge:verify 03` is a static check of the wiring; the live run is the evidence: graph, logs, states.

**Finished early?** What happens if the input is late? If the quality check fails? If the run is repeated? Show a log or output check for each answer.

| Symptom | Cause | Fix |
|---|---|---|
| `dags list-import-errors` shows a Cosmos error ending in an empty catalog name | Astro started before Databricks setup wrote the env | New terminal where `echo $DBT_CATALOG` prints a value, then `astro dev restart --wait 5m` |
| The freshness gate fails on a fresh seed | Connection error, not staleness: the task now says which | Same cold-start fix; then re-run the task |
| The failed gate stays yellow for minutes | `up_for_retry`: two retries at 2 and 4 minutes before it turns red | Wait about ten minutes, or set `retries: 0` while practising |
| `airflow: command not found` | The CLI lives in the containers | Prefix with `astro dev run` |
| `astro dev parse` says 0 DAGs | It does not mount data_products | Use the README's `python -c "import ast…"` check and `astro dev run dags list-import-errors` |
| `unpause` says `No paused DAGs were found` | The DAG was unpaused earlier; the state survives restarts | Nothing is wrong: go straight to `trigger` |
| `tasks clear` wants to delete dozens of tasks | The CLI clears by date, not by run | Answer no; clear from the UI, which clears the run you clicked |
| A BashOperator DAG is rejected | CG's cluster policy blocks arbitrary-shell operators | PythonOperator calling subprocess, as the starter does |
| `dbt docs serve` fails to bind | `DBT_HOST` is the Databricks host | `--host 127.0.0.1 --port 8081` |
| Two Astro environments on one laptop | Another project is running | `docker ps` and look for the devcontainer's scheduler |

### Walkthrough — Phase 2: production operations, on the price sheet

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#C9DAF8; color:#1a1a1a; padding:8px 14px; font-weight:bold;">🖥️ DEMO &nbsp;—&nbsp; Phase 2 on the price sheet</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">The ten Phase 2 challenges, one or two cells each, for when learners reach them.</div></div>

1. Use these cells at 13:00, or beside a learner who has reached Phase 2.
2. Every Phase 2 file ships complete: the task is to read each piece and prove it does what it says.
3. The failed run from step 10 is the evidence for challenges 1 to 4.

In [ ]:
%cd {HERE}/demos/airflow_local

#### P1 — The alert transport: one function, no vendor in the code

In [ ]:
!sed -n '/^def send_alert/,$p' dags/price_sheet/dags/notifications.py
# Every chat message goes through send_alert.

In [ ]:
!grep -niE 'https?://|slack_sdk|hooks.slack' dags/price_sheet/dags/notifications.py || echo 'no hard-coded URL and no vendor SDK: good'

#### P2, P3 — Three tiers: who hears about what

<img src="../common/diagrams/png/d4-alert-tiers.png" alt="Four events, four callbacks, and who hears about each" style="max-height:440px; max-width:100%;">

*Four events, four callbacks, and who hears about each*

In [ ]:
!grep -n '^def \|P[123]' dags/price_sheet/dags/callbacks.py | grep 'def \|"""'
# Each function, then the tier it serves: the two transports (Datadog, ticket) and the four callbacks.

In [ ]:
!bash run.sh alerts price_sheet_daily
# What the callbacks did in the failed run from step 10: a ticket for each retry (P3), then a Datadog event (P1) and a Teams message (P2).

#### P4 — Where each callback is attached

In [ ]:
!grep -n "_callback\|sla=" dags/price_sheet/dags/price_sheet_daily.py | grep -v ":    on_"
# Three places: default_args (every task), the DAG (the SLA miss), and the last working task (success once per run, and the SLA itself).

#### P5 — The freshness gate as a contract

In [ ]:
!bash run.sh dbt price_sheet source freshness --select source:vendor_feed
# The same check the gate runs.

#### P6 — The monitoring config: six panels

<img src="../common/diagrams/png/d4-monitoring.png" alt="Alerting says something broke; monitoring shows something drifting" style="max-height:380px; max-width:100%;">

*Alerting says something broke; monitoring shows something drifting*

In [ ]:
!grep -E 'name:|alert_condition:|alert_channel:' dags/price_sheet/include/monitoring/price_sheet.yml
# Each panel: a metric, when it alerts, and where the alert goes.

#### P7 — The runbook and the consumer guide

In [ ]:
!grep -n '^## \|^### Alert' dags/price_sheet/include/runbooks/price_sheet.md
# The runbook: what the on-call engineer opens when an alert lands.

In [ ]:
!sed -n '1,16p' dags/price_sheet/include/launch/consumer-guide.md
# The consumer guide: what a new user of the price sheet reads.

#### P8 — Structural tests: the DAG's shape, checked in milliseconds

In [ ]:
!python -m pytest dags/price_sheet/tests -v --no-header 2>&1 | grep -E 'PASSED|FAILED|passed|failed'
# No Airflow and no warehouse: the files parse, the chain is wired, the callbacks and the SLA are attached, and no file holds a secret.

#### P9 — Stretch: the other shape, dbt inside Databricks

<img src="../common/diagrams/png/d4-two-shapes.png" alt="Two ways to run dbt from Airflow: on the worker (Cosmos), or inside Databricks (CG's paved path)" style="max-height:420px; max-width:100%;">

*Two ways to run dbt from Airflow: on the worker (Cosmos), or inside Databricks (CG's paved path)*

In [ ]:
!sed -n '1,16p' ../../../../reference/DataCreators-client/dev-container/edo-training-data-products/challenges/v1/03-airflow-orchestration/solution/dags/databricks_submit_run_dag.py
# The challenge's second DAG: the same pipeline, as jobs submitted to Databricks.

#### P10 — Capstone: the production checklist

| Check | On the price sheet | Where you saw it |
|---|---|---|
| Schedule | `0 6 * * 1-5`, `Europe/London`, `catchup=False`, `max_active_runs=1` | step 7 |
| Fail fast | `source_freshness_check` runs first | steps 4 and 10 |
| Build | the Cosmos `DbtTaskGroup` | step 5 |
| Retries | `retries=2` with backoff | step 8 |
| Tiered alerting | failure and retry in `default_args`, SLA miss on the DAG | P2 to P4 |
| SLA | `publish_to_catalog` carries `sla=timedelta(minutes=90)` | P4 |
| Documents | runbook, consumer guide, monitoring config | P6, P7 |
| No secrets | no token or host in any DAG file | P8 |

In [ ]:
%cd {HERE}

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#EFEFEF; color:#1a1a1a; padding:8px 14px; font-weight:bold;">🍽️ LUNCH &nbsp;·&nbsp; 60 min &nbsp;·&nbsp; back at 13:00</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">Questions go in the table at the end; the next part starts on time.</div></div>

### Shareback — one run, one log line, one output check per pair

<div style="border:2px solid #F8DB59; border-radius:8px; margin:10px 0;">
<div style="background:#FFF2CC; color:#000; padding:8px 14px; font-weight:bold;">💭 SHAREBACK — 13:30, one pair at a time</div>
<div style="background:#fffef5; padding:12px 14px; color:#1a1a1a;">Show the run ID, the log line that told you what happened, and the table it produced.</div></div>

- Keep credential and path errors separate from the intended data-quality failure.
- Fast finishers: what happens if the input is late? If the quality check fails? If the run is repeated?

<a id="s3"></a>

## 3. CI/CD and release operations

### Demo — GitOps, Harness, Datadog

<div style="border:2px solid #434343; border-radius:8px; margin:18px 0 12px; overflow:hidden;">
<div style="background:#434343; color:#fff; padding:8px 14px; font-weight:bold;">🧭 WHAT, WHY AND HOW</div>
<div style="background:#F3F3F3; padding:10px 14px; color:#1a1a1a; font-style:italic;">Your DAG and dbt project are in Git. How do they get to production, and who presses the button?</div>
<div style="display:flex; gap:8px; padding:10px; background:#fff;"><div style="flex:1; min-width:0; background:#D0E0E3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHAT</div>Continuous integration (merge often) and continuous deployment (release often), driven by GitHub events through Harness, into Astro and Databricks, watched by Datadog.</div><div style="flex:1; min-width:0; background:#FFF2CC; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHY</div>Creators rarely operate this layer, but they read it: which pipeline ran, where the logs are, who deployed last night.</div><div style="flex:1; min-width:0; background:#D9EAD3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">HOW</div>The GitOps picture; a Harness walkthrough (guest, screenshots, or the anchor); monitors as the SLIs made visible.</div></div></div>

<img src="../common/diagrams/png/d4-cicd.png" alt="From a branch to production: pull request, merge, Harness, SQA, production with a change record" style="max-height:320px; max-width:100%;">

**Think:** What do CI and CD stand for, and where is each one in this picture?

**Answers from the room:**

- 

<img src="../common/diagrams/png/d4-cicd-tradeoffs.png" alt="What CI/CD gives, and what it costs" style="max-height:460px; max-width:100%;">

**Think:** What does CI/CD give a team, and what does it cost?

**Answers from the room:**

- 

<img src="../common/diagrams/png/d4-harness.png" alt="Harness: the creator opens and merges a pull request; Harness does the rest" style="max-height:340px; max-width:100%;">

**Think:** What is Harness here, and how much of it does a creator touch?

**Answers from the room:**

- 

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#C9DAF8; color:#1a1a1a; padding:8px 14px; font-weight:bold;">🖥️ DEMO &nbsp;—&nbsp; Harness, with screenshots or a guest</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">Walk a deployment to SQA and then to production: live if someone has access, else from the pre-flight screenshots.</div></div>

1. Pick the environment and find the repository's pipeline.
2. SQA: Run, apply the SQA input set, choose the feature branch, run; a few minutes.
3. Check who deployed to SQA last and whether they finished testing: SQA is shared.
4. Production: the PRD pipeline, fixed values, the release branch, and a change record (CRQ) number before anything runs.
5. Open the logs of a failed execution.

In [ ]:
!grep -E 'name:|alert_condition:|alert_channel:' demos/airflow_local/dags/price_sheet/include/monitoring/price_sheet.yml
# The price sheet's monitoring config again: six panels, each with an alert condition and a channel.

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#EFEFEF; color:#1a1a1a; padding:8px 14px; font-weight:bold;">☕ BREAK &nbsp;·&nbsp; 15 min &nbsp;·&nbsp; back at 14:15</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">Questions go in the table at the end; the next part starts on time.</div></div>

### Activity 10 — release decision

<div style="border:2px solid #F8DB59; border-radius:8px; margin:10px 0;">
<div style="background:#F8DB59; color:#000; padding:8px 14px; font-weight:bold;">🎯 ACTIVITY 10 &nbsp;—&nbsp; Release decision &nbsp;·&nbsp; Pairs &nbsp;·&nbsp; 20 min</div>
<div style="background:#fffef5; padding:12px 14px; color:#1a1a1a;"><strong>The nightly build produced a candidate output with 500 rows but only 497 distinct IDs, so the uniqueness check failed and the candidate was not published. Yesterday's accepted output is still available to consumers. In ten minutes, decide what should happen next and draft a two-sentence update to the consumer. Then ten minutes of discussion.</strong><br><br><strong>Produce:</strong><br>• A decision: what is released, what stays blocked, and who decides<br>• A two-sentence update to the consumer<br>• The information you are missing and who can supply it<br><br><strong>Debrief questions:</strong><br>• Does the consumer's freshness commitment allow yesterday's output to stand today?<br>• What would make the three duplicate IDs a data problem rather than a pipeline problem?<br>• Who gets to override a failed quality gate, and how is that recorded?</div></div>

### Activity 11 — explain a failure and repair

<div style="border:2px solid #F8DB59; border-radius:8px; margin:10px 0;">
<div style="background:#F8DB59; color:#000; padding:8px 14px; font-weight:bold;">🎯 ACTIVITY 11 &nbsp;—&nbsp; Explain a failure and repair &nbsp;·&nbsp; Pairs, then swap &nbsp;·&nbsp; 15 min</div>
<div style="background:#fffef5; padding:12px 14px; color:#1a1a1a;"><strong>Write a short note about the Airflow failure and repair you ran today (or the prepared example if your environment was blocked): the affected run, the useful error, the cause, the action taken, the output check, and anything still unresolved. Swap notes with another pair for five minutes: could they locate the run and explain what happened from the note alone?</strong><br><br><strong>Produce:</strong><br>• The note, using the template below<br>• The other pair's verdict: could they find the run?<br><br><strong>Debrief questions:</strong><br>• Which line of the note was the most useful to the reader?<br>• What did you have to go back to the UI to find?</div></div>

<details><summary><strong>Prepared failure example</strong> (for pairs whose environment was blocked)</summary>

Expected: the freshness gate fails because the custodian feed is older than its `error_after` threshold; `dbt_pipeline` and `publish_to_catalog` are `upstream_failed`; no rows are published; the previous run's output stands. Observed in the demo stack: the `source_freshness_check` log ends with "dbt source freshness failed: the custodian feed breached its error_after threshold"; the repair was to refresh the feed and trigger a new run, which published eight rows. Distinguish this from a credentials or path error, whose first useful line names a missing environment value or executable.

</details>

### Friday preparation — the checklist, then the close

<div style="border:2px solid #F8DB59; border-radius:8px; margin:10px 0;">
<div style="background:#FFF2CC; color:#000; padding:8px 14px; font-weight:bold;">💭 FRIDAY CHECKLIST — every learner, before leaving</div>
<div style="background:#fffef5; padding:12px 14px; color:#1a1a1a;">Five checks; any failure goes on the list with a name and a time it will be fixed.</div></div>

- ☐ Open `challenges/v1/04-atlas-capstone/README.md` and read the plan of the day.
- ☐ From the repo root: `test -f .mise.toml && test -d data_products/atlas_elt && echo OK`.
- ☐ `mise exec -- databricks auth describe --profile DEFAULT`.
- ☐ From `data_products/atlas_elt`: `dbt debug` passes.
- ☐ `astro dev stop`, so tomorrow starts clean.

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#D9EAD3; color:#1a1a1a; padding:8px 14px; font-weight:bold;">✅ BEFORE LEARNERS LEAVE</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">☐ The DAG, run and task identities, useful logs, an output check and a short explanation of the failure and repair, per pair.<br>☐ Every learner has opened challenges/v1/04-atlas-capstone/README.md and run the readiness checks in data_products/atlas_elt.<br>☐ A list of setup issues that could stop Friday's start, each with the person responsible for the next action.<br>☐ The Airflow stack stopped on the learners' laptops (astro dev stop) so Friday starts clean.</div></div>

<a id="questions"></a>

## Questions from the room

Type the question in the table as it comes; answer live in the cells below; tidy up at the break. Anything unanswered goes to the platform team or the Teams channel, with the person who asked.

| # | Question | Who asked | Answer / where it went |
|---|---|---|---|
| 1 |  |  |  |
| 2 |  |  |  |
| 3 |  |  |  |

In [ ]:
# Answer live here: SQL against a demo warehouse, a small Python check, or a shell command.
# Example: sql("select * from training_demo.stg_holdings limit 5", HERE / "demos/portfolio_analytics/warehouse/training_dev.duckdb")

In [ ]:
# Sketch a diagram live (before Excalidraw). Edit and run.
quick_diagram("""
flowchart LR
  A[source file] --> B[dbt rules on Databricks] --> C[consumer table]
""")

**Answers written up after the day** — fill in here, then share the notebook.

- 

---
<small>&copy; 2026 Decoded Limited. All rights reserved. [decoded.com](https://decoded.com)</small>